# Get results out of my WandB automatically

In [1]:
import wandb
import numpy as np
from itertools import product

api = wandb.Api()
runs = api.runs("labeebah-islaam/world_models")  # <-- replace this

In [4]:
modes = ["reward", "value"]
steps = [0, 1, 2, 5, 10, 15, 20]
thresholds = [2, 1.5, 1]
inner_steps = [0, 1, 2, 5]

metric = "actor_critic/eval/planned_cumulative_reward"

def get_last_value(run):
    try:
        hist = run.history(keys=[metric], pandas=True)
        if not hist.empty:
            return hist[metric].dropna().values[-1]
    except Exception:
        pass
    return None

results = []

for mode, step, thres, inner in product(modes, steps, thresholds, inner_steps):
    # print(f"Processing mode={mode}, steps={step}, thres={thres}, inner={inner}")
    matched = []
    for run in runs:
        cfg = run.config
        if run.state != "finished":
            continue
        try:
            if (
                str(cfg["evaluation"]["planning_mode"]) == str(mode) and
                int(cfg["evaluation"]["planning_steps"]) == int(step) and
                float(cfg["evaluation"]["entropy_threshold"]) == float(thres) and
                int(cfg["evaluation"]["inner_planning_steps"]) == int(inner)
            ):
                val = get_last_value(run)
                if val is not None:
                    matched.append(val)
        except KeyError:
            # If the config does not have the expected keys, skip this run
            continue

    if len(matched) >= 3:
        print(f"Matched {len(matched)} runs for mode={mode}, steps={step}, thres={thres}, inner={inner}")
        arr = np.array(matched[:3])  # only take first 3
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "mean": arr.mean(),
            "std": arr.std()
        }
    else:
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "mean": None,
            "std": None
        }

    results.append(result)

# Print or export
import pandas as pd
df = pd.DataFrame(results)
print(df)
df.to_csv("sweep_results.csv", index=False)

Matched 6 runs for mode=reward, steps=1, thres=2, inner=1
Matched 3 runs for mode=reward, steps=1, thres=2, inner=2
Matched 3 runs for mode=reward, steps=1, thres=2, inner=5
Matched 7 runs for mode=reward, steps=1, thres=1.5, inner=1
Matched 6 runs for mode=reward, steps=1, thres=1.5, inner=2
Matched 5 runs for mode=reward, steps=1, thres=1.5, inner=5
Matched 3 runs for mode=reward, steps=2, thres=2, inner=1
Matched 3 runs for mode=reward, steps=5, thres=2, inner=1
Matched 3 runs for mode=reward, steps=5, thres=2, inner=2
Matched 3 runs for mode=reward, steps=5, thres=2, inner=5
Matched 3 runs for mode=reward, steps=10, thres=2, inner=1
Matched 3 runs for mode=reward, steps=10, thres=2, inner=2
Matched 3 runs for mode=reward, steps=10, thres=2, inner=5
Matched 3 runs for mode=value, steps=0, thres=1.5, inner=0
Matched 3 runs for mode=value, steps=1, thres=2, inner=1
Matched 3 runs for mode=value, steps=1, thres=2, inner=2
Matched 3 runs for mode=value, steps=1, thres=2, inner=5
Matched

In [27]:
for run in runs[0].config:
    print(run)

print(runs[0].config)

env
agent
wandb
common
denoiser
training
collection
evaluation
actor_critic
checkpointing
rew_end_model
initialization
static_dataset
world_model_env
{'env': {'test': {'id': 'BoxingNoFrameskip-v4', 'size': 64, 'done_on_life_loss': False, 'max_episode_steps': None}, 'train': {'id': 'BoxingNoFrameskip-v4', 'size': 64, 'done_on_life_loss': True, 'max_episode_steps': None}, 'keymap': 'atari/BoxingNoFrameskip-v4'}, 'agent': {'_target_': 'agent.AgentConfig', 'denoiser': {'_target_': 'models.diffusion.DenoiserConfig', 'sigma_data': 0.5, 'inner_model': {'depths': [2, 2, 2, 2], '_target_': 'models.diffusion.InnerModelConfig', 'channels': [64, 64, 64, 64], 'attn_depths': [0, 0, 0, 0], 'img_channels': 3, 'cond_channels': 256, 'num_steps_conditioning': 4}, 'sigma_offset_noise': 0.3}, 'actor_critic': {'down': [1, 1, 1, 1], '_target_': 'models.actor_critic.ActorCriticConfig', 'channels': [32, 32, 64, 64], 'img_size': 64, 'lstm_dim': 512, 'img_channels': 3}, 'rew_end_model': {'depths': [2, 2, 2, 2], 

In [48]:
# print(cfg.get("evaluation", {}).get("planning_steps"), type(cfg.get("evaluation", {}).get("planning_steps")))
print(runs[100].config['evaluation']['inner_planning_steps'])

1
